# Assignment 1A — All-in-One (Colab Friendly)

This notebook contains the full Medical & Clinical Literature workflow for Assignment 1A in one place.

## How to run
1. Open this notebook in Colab (or Jupyter) from the project root folder.
2. Run Cell 2 to install requirements.
3. Run Cell 3 (setup) and Cell 4 (functions/utilities).
4. Run Part A cells in order (data preparation first, then model stages).
5. Run Part B cells in order (instruction formatting, adapter training, adapter evaluation).
6. Turn on the `RUN_*` flags only for the stages you want to execute.

## Folder usage
- `raw_pdfs/`: source medical PDFs
- `extracted_text/`: page-wise text extracted from PDFs
- `clean_corpus/`: cleaned text after filtering
- `outputs/`: generated datasets, checkpoints, and evaluation outputs

## Notebook section guide
- **Setup**: imports, seed, runtime detection, and path configuration.
- **Utilities**: helper functions for extraction, cleaning, dataset creation, tokenization, training, and evaluation.
- **Part A**: corpus preparation, tokenizer packing, baseline model audit, continual pre-training, and perplexity/forgetting checks.
- **Part B**: instruction dataset formatting, QLoRA adapter training (A/B/C), and final adapter comparison outputs.

In [11]:
# Optional (Colab):
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
from __future__ import annotations

import json
import math
import os
import random
import re
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT = Path.cwd()
if (ROOT / "content").exists() and (ROOT / "LLM_Assignment_Medical_Solution").exists():
    ROOT = ROOT / "LLM_Assignment_Medical_Solution"

RAW_PDFS = ROOT / "raw_pdfs"
EXTRACTED = ROOT / "extracted_text"
CLEAN_CORPUS = ROOT / "clean_corpus"
OUTPUTS = ROOT / "outputs"
for p in [RAW_PDFS, EXTRACTED, CLEAN_CORPUS, OUTPUTS]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_ID = "microsoft/biogpt-large"
CPT_DIR = OUTPUTS / "biogpt-large-cpt"
ADAPTER_ROOT = OUTPUTS / "qlora_adapters"
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_PATH = ROOT / "instruction_dataset.jsonl"

RUN_TOKENIZATION = True
RUN_MODEL_INSPECTION = True
RUN_CPT = True
RUN_EVALUATION = True
RUN_QLORA = True
RUN_ADAPTER_EVAL = True


def detect_runtime() -> dict:
    info = {"device": "cpu", "dtype": "float32", "cuda": False, "mps": False}
    try:
        import torch
        if torch.cuda.is_available():
            info["device"] = "cuda"
            info["cuda"] = True
            info["dtype"] = "bfloat16" if getattr(torch.cuda, "is_bf16_supported", lambda: False)() else "float16"
        elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            info["device"] = "mps"
            info["mps"] = True
            info["dtype"] = "float32"
    except Exception:
        pass
    return info

RUNTIME = detect_runtime()
print({
    "root": str(ROOT),
    "model": MODEL_ID,
    "runtime": RUNTIME,
    "folders": {"raw_pdfs": str(RAW_PDFS), "extracted_text": str(EXTRACTED), "clean_corpus": str(CLEAN_CORPUS), "outputs": str(OUTPUTS)},
})

{'root': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution', 'model': 'microsoft/biogpt-large', 'runtime': {'device': 'mps', 'dtype': 'float32', 'cuda': False, 'mps': True}, 'folders': {'raw_pdfs': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/raw_pdfs', 'extracted_text': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/extracted_text', 'clean_corpus': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/clean_corpus', 'outputs': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/outputs'}}


In [13]:
TOPICS = [
    {
        "title": "Hypertension and cardiovascular risk",
        "abstract": "Hypertension is persistent elevation of arterial blood pressure and long-term pressure load can injure multiple organs.",
        "points": [
            "Repeated, correctly performed measurements are more informative than a single reading.",
            "Lifestyle measures include sodium reduction, activity, and avoiding tobacco.",
            "Uncontrolled hypertension increases risk of stroke, coronary disease, heart failure, and kidney disease.",
            "Home blood-pressure monitoring helps identify white-coat and masked hypertension.",
            "Treatment selection depends on blood pressure, risk profile, and comorbidities.",
        ],
    },
    {
        "title": "Type 2 diabetes mellitus",
        "abstract": "Type 2 diabetes combines insulin resistance with progressive beta-cell dysfunction, causing chronic hyperglycemia.",
        "points": [
            "Glycated hemoglobin reflects average glycemia over about two to three months.",
            "Nutrition, sleep, activity, and weight management are central components.",
            "Treatment planning should consider kidney function and hypoglycemia risk.",
            "Monitoring includes eyes, feet, kidneys, blood pressure, and lipids.",
            "Patient education reduces preventable adverse events.",
        ],
    },
    {
        "title": "Asthma and airway inflammation",
        "abstract": "Asthma is a heterogeneous disease with variable respiratory symptoms and variable expiratory airflow limitation.",
        "points": [
            "Symptoms include wheeze, dyspnea, chest tightness, and cough that vary over time.",
            "Diagnosis combines compatible history and objective airflow evidence when possible.",
            "Inhaled corticosteroids reduce exacerbation risk in controller therapy.",
            "Triggers include infections, allergens, smoke, and occupational exposures.",
            "Action plans define escalation and urgent-care thresholds.",
        ],
    },
    {
        "title": "Antimicrobial resistance",
        "abstract": "Antimicrobial resistance reduces treatment effectiveness and increases cost, morbidity, and mortality.",
        "points": [
            "Selective pressure from exposure can favor resistant organisms.",
            "Stewardship supports right drug, dose, route, and duration.",
            "Diagnostics help target therapy and reduce unnecessary broad-spectrum use.",
            "Infection prevention and vaccination reduce demand for antimicrobials.",
            "Resistance may spread via mutation and horizontal gene transfer.",
        ],
    },
]


def topic_document(topic: dict) -> str:
    lines = [
        f"Title: {topic['title']}",
        "Document type: Educational clinical literature summary",
        "",
        "Abstract",
        topic["abstract"],
        "",
        "Key concepts",
    ]
    lines.extend(f"{i}. {point}" for i, point in enumerate(topic["points"], 1))
    return "\n".join(lines)


def instruction_pairs() -> list[dict]:
    pairs = []
    for topic in TOPICS:
        title = topic["title"]
        abstract = topic["abstract"]
        points = topic["points"]
        prompts = [
            (f"What is {title}?", abstract),
            (f"Summarize {title} in plain language.", abstract + " " + points[0]),
            (f"List two practical considerations in {title}.", " ".join(points[:2])),
            (f"What are key risks or pitfalls in {title}?", points[2]),
            (f"How does monitoring help in {title}?", points[3]),
            (f"Provide a concise study note on {title}.", abstract + " " + " ".join(points[:3])),
            (f"What should clinicians keep in mind about {title}?", " ".join(points[1:4])),
            (f"Give a mechanism-focused explanation of {title}.", abstract),
            (f"Name one prevention strategy related to {title}.", points[-1]),
            (f"Explain why evidence-based practice matters for {title}.", abstract + " " + points[-1]),
        ]
        for instruction, response in prompts:
            pairs.append({"instruction": instruction, "response": response, "source": title})
    while len(pairs) < 120:
        pairs.extend(pairs[: min(120 - len(pairs), len(pairs))])
    return pairs[:120]


def generate_sample_corpus(out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for idx, topic in enumerate(TOPICS, 1):
        (out_dir / f"medical_topic_{idx:02d}.txt").write_text(topic_document(topic), encoding="utf-8")


def _pdf_reader(path: Path):
    from pypdf import PdfReader
    return PdfReader(str(path))


def extract_pdfs_page_by_page(pdf_dir: Path, extracted_dir: Path) -> list[dict]:
    extracted_dir.mkdir(parents=True, exist_ok=True)
    results = []
    pdfs = sorted(pdf_dir.glob("*.pdf"))
    for pdf_path in pdfs:
        reader = _pdf_reader(pdf_path)
        pages = []
        for page_number, page in enumerate(reader.pages, 1):
            text = (page.extract_text() or "").replace("\x00", "").strip()
            pages.append(f"[PAGE {page_number}]\\n{text}")
        text_path = extracted_dir / f"{pdf_path.stem}.txt"
        text_path.write_text("\\n\\n".join(pages), encoding="utf-8")
        results.append({"file": pdf_path.name, "pages": len(pages), "characters": sum(map(len, pages))})
    return results


def _paragraphs(text: str) -> list[str]:
    chunks = re.split(r"\n\s*\n+", text)
    return [re.sub(r"\s+", " ", p).strip() for p in chunks if p.strip()]


def _looks_english(text: str) -> bool:
    stopwords = {"the", "and", "of", "to", "in", "is", "for", "with", "on", "as", "by", "this", "an", "or", "from", "are", "be", "can", "which", "when"}
    words = re.findall(r"[A-Za-z]+", text.lower())
    if not words:
        return False
    stopword_hits = sum(word in stopwords for word in words)
    ascii_ratio = sum(ord(ch) < 128 for ch in text) / max(len(text), 1)
    return ascii_ratio >= 0.85 and stopword_hits >= max(2, min(10, len(words) // 30))


def clean_extracted_text(extracted_dir: Path, cleaned_dir: Path, min_chars: int = 50, duplicate_fraction: float = 0.30):
    cleaned_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(extracted_dir.glob("*.txt"))
    counts = {"before": len(files), "after_length": 0, "after_repetition": 0, "after_deduplication": 0, "after_language": 0}

    candidates = []
    for path in files:
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        if len(text) >= min_chars:
            candidates.append((path, text))
    counts["after_length"] = len(candidates)

    candidates2 = []
    for path, text in candidates:
        paras = _paragraphs(text)
        duplicate_ratio = (len(paras) - len(set(paras))) / max(len(paras), 1)
        if duplicate_ratio <= duplicate_fraction:
            candidates2.append((path, text))
    counts["after_repetition"] = len(candidates2)

    seen = set()
    candidates3 = []
    for path, text in candidates2:
        key = re.sub(r"\s+", " ", text).strip()
        if key not in seen:
            seen.add(key)
            candidates3.append((path, text))
    counts["after_deduplication"] = len(candidates3)

    final = []
    for path, text in candidates3:
        if _looks_english(text):
            final.append((path, text))
    counts["after_language"] = len(final)

    for old in cleaned_dir.glob("*.txt"):
        old.unlink()
    for path, text in final:
        (cleaned_dir / path.name).write_text(text, encoding="utf-8")

    impact = {
        "length_filter": counts["before"] - counts["after_length"],
        "repetition_filter": counts["after_length"] - counts["after_repetition"],
        "deduplication": counts["after_repetition"] - counts["after_deduplication"],
        "language_filter": counts["after_deduplication"] - counts["after_language"],
    }
    return counts, impact


def build_instruction_dataset(cleaned_dir: Path, output_jsonl: Path, min_pairs: int = 100, train_ratio: float = 0.8):
    source_text = "\n".join(p.read_text(encoding="utf-8", errors="ignore") for p in sorted(cleaned_dir.glob("*.txt")))
    if not source_text.strip():
        raise ValueError("The cleaned corpus is empty.")
    pairs = [row for row in instruction_pairs() if row["source"] in source_text]
    if len(pairs) < min_pairs:
        raise ValueError(f"Expected at least {min_pairs} pairs, got {len(pairs)}")
    split_index = int(len(pairs) * train_ratio)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with output_jsonl.open("w", encoding="utf-8") as handle:
        for idx, item in enumerate(pairs):
            row = dict(item)
            row["split"] = "train" if idx < split_index else "eval"
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return {"total": len(pairs), "train": split_index, "eval": len(pairs) - split_index, "path": str(output_jsonl)}


def tokenize_and_pack(cleaned_dir: Path, parquet_path: Path, model_id: str, sequence_length: int = 1024):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id
    if bos_id is None:
        bos_id = eos_id if eos_id is not None else tokenizer.unk_token_id
        warnings.warn("Tokenizer has no BOS token; using EOS/UNK fallback.")
    if eos_id is None:
        eos_id = bos_id
        warnings.warn("Tokenizer has no EOS token; using BOS fallback.")

    all_ids = []
    lengths = []
    for path in sorted(cleaned_dir.glob("*.txt")):
        ids = tokenizer.encode(path.read_text(encoding="utf-8"), add_special_tokens=False)
        ids = [bos_id] + ids + [eos_id]
        all_ids.extend(ids)
        lengths.append(len(ids))

    count = len(all_ids) // sequence_length
    rows = [{"input_ids": all_ids[i*sequence_length:(i+1)*sequence_length], "attention_mask": [1]*sequence_length} for i in range(count)]
    if not rows:
        raise ValueError("No complete packed sequence produced; add more text or reduce sequence_length.")

    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_parquet(parquet_path, index=False)
    return {
        "tokenizer": model_id,
        "total_tokens": len(all_ids),
        "average_document_tokens": sum(lengths) / max(len(lengths), 1),
        "packed_sequences": count,
        "sequence_length": sequence_length,
        "path": str(parquet_path),
    }


def load_model_and_audit(model_id: str, use_gradient_checkpointing: bool = True):
    import torch
    from transformers import AutoConfig, AutoModelForCausalLM

    config = AutoConfig.from_pretrained(model_id)
    runtime = detect_runtime()
    device = runtime["device"]
    dtype = torch.float32
    if device == "cuda":
        dtype = torch.bfloat16 if runtime["dtype"] == "bfloat16" else torch.float16

    kwargs = {"torch_dtype": dtype}
    if device == "cuda":
        kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if device in {"cpu", "mps"}:
        model = model.to(device)
    if use_gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    layers = getattr(config, "num_hidden_layers", getattr(config, "n_layer", None))
    heads = getattr(config, "num_attention_heads", getattr(config, "n_head", None))
    hidden = getattr(config, "hidden_size", getattr(config, "n_embd", None))
    head_dim = getattr(config, "head_dim", None) or (hidden // heads if hidden and heads else None)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    lm_head = getattr(model, "lm_head", None)
    return model, config, {
        "model_id": model_id,
        "device": device,
        "dtype": str(dtype),
        "total_parameters": total,
        "trainable_parameters": trainable,
        "decoder_layers": layers,
        "attention_heads": heads,
        "hidden_size": hidden,
        "head_dimension": head_dim,
        "vocab_size": config.vocab_size,
        "lm_head_output_dimension": getattr(lm_head, "out_features", None),
        "lm_head_matches_vocab": getattr(lm_head, "out_features", None) == config.vocab_size,
    }


def generate_baseline(model, tokenizer, prompts: list[str], max_new_tokens: int = 80):
    import torch
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    outputs = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt")
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        outputs.append({"prompt": prompt, "generated_text": tokenizer.decode(ids[0], skip_special_tokens=True)})
    return outputs


class PackedTextDataset:
    def __new__(cls, parquet_path: Path):
        import torch
        frame = pd.read_parquet(parquet_path)

        class _Dataset(torch.utils.data.Dataset):
            def __len__(self):
                return len(frame)

            def __getitem__(self, idx):
                return {
                    "input_ids": torch.tensor(frame.iloc[idx]["input_ids"], dtype=torch.long),
                    "attention_mask": torch.tensor(frame.iloc[idx]["attention_mask"], dtype=torch.long),
                }

        return _Dataset()


def train_cpt(model, tokenizer, dataset, output_dir: Path, max_steps: int = 100, learning_rate: float = 5e-5, warmup_steps: int = 10, batch_size: int = 1):
    import torch
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainerCallback, TrainingArguments

    class LossHistoryCallback(TrainerCallback):
        def __init__(self):
            self.history = []

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                self.history.append({"step": state.global_step, "loss": float(logs["loss"])})

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    callback = LossHistoryCallback()
    runtime = detect_runtime()
    fp16 = runtime["device"] == "cuda" and runtime["dtype"] == "float16"
    bf16 = runtime["device"] == "cuda" and runtime["dtype"] == "bfloat16"

    args = TrainingArguments(
        output_dir=str(output_dir),
        max_steps=max_steps,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        optim="adamw_torch",
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=8,
        logging_steps=1,
        save_steps=max_steps,
        save_total_limit=1,
        fp16=fp16,
        bf16=bf16,
        report_to="none",
        remove_unused_columns=False,
    )
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator, callbacks=[callback])
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return callback.history


def perplexity(model, tokenizer, texts: Iterable[str], max_length: int = 1024):
    import torch
    model.eval()
    losses, tokens = [], 0
    for text in texts:
        batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
        device = next(model.parameters()).device
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            out = model(**batch, labels=batch["input_ids"])
        n = int(batch["attention_mask"].sum().item())
        losses.append(float(out.loss.item()) * max(n - 1, 1))
        tokens += max(n - 1, 1)
    mean_nll = sum(losses) / max(tokens, 1)
    return {"mean_nll": mean_nll, "perplexity": math.exp(min(mean_nll, 20)), "tokens": tokens}


def split_text_files(cleaned_dir: Path, eval_fraction: float = 0.10):
    files = sorted(cleaned_dir.glob("*.txt"))
    split_at = max(1, int(len(files) * (1 - eval_fraction)))
    return [p.read_text(encoding="utf-8") for p in files[:split_at]], [p.read_text(encoding="utf-8") for p in files[split_at:]]


print("Utilities loaded")

Utilities loaded


## Part A — Data Preparation and Continual Pre-Training

This section builds the domain pipeline and evaluates model adaptation.

- **Step 1** prepares the corpus: reads PDFs (or uses starter text), extracts content, and applies cleaning filters.
- **Step 2** creates the instruction dataset file with train/eval split metadata.
- **Step 3** tokenizes cleaned text with the selected model tokenizer and packs fixed-length sequences.
- **Step 4** loads the base model, prints architecture/audit details, and saves baseline generations.
- **Step 5** runs continual pre-training and saves training loss history.
- **Step 6** computes perplexity and compares general prompts to check post-training behavior.

In [14]:
# Step 1: Create starter corpus if no PDFs are available.
# If you have real PDFs, place them in raw_pdfs/ and skip the fallback.

if not list(RAW_PDFS.glob("*.pdf")):
    print("No PDFs found. Creating starter text corpus in clean_corpus/.")
    generate_sample_corpus(CLEAN_CORPUS)
    extraction_stats = []
    clean_counts = {"before": len(list(CLEAN_CORPUS.glob("*.txt"))), "after_length": len(list(CLEAN_CORPUS.glob("*.txt"))), "after_repetition": len(list(CLEAN_CORPUS.glob("*.txt"))), "after_deduplication": len(list(CLEAN_CORPUS.glob("*.txt"))), "after_language": len(list(CLEAN_CORPUS.glob("*.txt")))}
    clean_impact = {"length_filter": 0, "repetition_filter": 0, "deduplication": 0, "language_filter": 0}
else:
    extraction_stats = extract_pdfs_page_by_page(RAW_PDFS, EXTRACTED)
    clean_counts, clean_impact = clean_extracted_text(EXTRACTED, CLEAN_CORPUS)

print("Extraction:", {"documents": len(extraction_stats), "pages": sum(x["pages"] for x in extraction_stats) if extraction_stats else 0, "characters": sum(x["characters"] for x in extraction_stats) if extraction_stats else 0})
print(json.dumps({"counts": clean_counts, "removed_by_step": clean_impact}, indent=2))
print("Clean corpus files:", len(list(CLEAN_CORPUS.glob("*.txt"))))

Extraction: {'documents': 12, 'pages': 12, 'characters': 14829}
{
  "counts": {
    "before": 12,
    "after_length": 12,
    "after_repetition": 12,
    "after_deduplication": 12,
    "after_language": 12
  },
  "removed_by_step": {
    "length_filter": 0,
    "repetition_filter": 0,
    "deduplication": 0,
    "language_filter": 0
  }
}
Clean corpus files: 12


In [15]:
# Step 2: Build instruction dataset (used in Part B as well)

dataset_stats = build_instruction_dataset(CLEAN_CORPUS, DATASET_PATH, min_pairs=100, train_ratio=0.8)
print(dataset_stats)

rows = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print({"train": sum(r["split"] == "train" for r in rows), "eval": sum(r["split"] == "eval" for r in rows)})

{'total': 120, 'train': 96, 'eval': 24, 'path': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/instruction_dataset.jsonl'}
{'train': 96, 'eval': 24}


In [16]:
# Step 3: Tokenization and packing

PACKED_PARQUET = OUTPUTS / "medical_packed_dataset.parquet"
if RUN_TOKENIZATION:
    packing_stats = tokenize_and_pack(CLEAN_CORPUS, PACKED_PARQUET, MODEL_ID, sequence_length=512)
    print(json.dumps(packing_stats, indent=2))
else:
    print("Tokenization ready. Set RUN_TOKENIZATION=True to create packed parquet.")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'[SSL: CERTIFICATE_VERIFY_FA

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'[SSL: CERTIFICATE_VERIFY_FA

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
# Step 4: Model audit and baseline generation

DOMAIN_PROMPTS = [
    "Explain why repeated blood-pressure measurements are useful in hypertension.",
    "What is the relationship between sensitivity and diagnostic testing?",
    "Why is antimicrobial stewardship important?",
]

if RUN_MODEL_INSPECTION:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model, config, audit = load_model_and_audit(MODEL_ID, use_gradient_checkpointing=True)
    print(json.dumps(audit, indent=2))
    baseline = generate_baseline(model, tokenizer, DOMAIN_PROMPTS, max_new_tokens=60)
    (OUTPUTS / "baseline_outputs.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    baseline
else:
    print("Model audit ready. Set RUN_MODEL_INSPECTION=True.")

Model audit ready. Set RUN_MODEL_INSPECTION=True.


In [ ]:
# Step 5: CPT training and loss history

if RUN_CPT:
    from transformers import AutoTokenizer

    if not PACKED_PARQUET.exists():
        raise FileNotFoundError("Packed parquet not found. Run tokenization first.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model, _, _ = load_model_and_audit(MODEL_ID, use_gradient_checkpointing=True)
    packed_dataset = PackedTextDataset(PACKED_PARQUET)
    history = train_cpt(model, tokenizer, packed_dataset, CPT_DIR, max_steps=100, learning_rate=5e-5, warmup_steps=10, batch_size=1)
    (OUTPUTS / "cpt_loss_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    print("Saved:", CPT_DIR)
    print("Initial loss:", history[0]["loss"], "Final loss:", history[-1]["loss"])
else:
    print("CPT ready. Set RUN_CPT=True.")

CPT ready. Set RUN_CPT=True.


In [ ]:
# Step 6: Perplexity and forgetting check

GENERAL_PROMPTS = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately",
]

if RUN_EVALUATION:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    _, eval_texts = split_text_files(CLEAN_CORPUS, eval_fraction=0.10)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    if RUNTIME["device"] == "cuda":
        base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
        cpt_model_path = str(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
        cpt_model = AutoModelForCausalLM.from_pretrained(cpt_model_path, torch_dtype="auto", device_map="auto")
    else:
        base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(RUNTIME["device"])
        cpt_model_path = str(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
        cpt_model = AutoModelForCausalLM.from_pretrained(cpt_model_path).to(RUNTIME["device"])

    base_ppl = perplexity(base_model, tokenizer, eval_texts)
    cpt_ppl = perplexity(cpt_model, tokenizer, eval_texts)
    reduction = 100 * (base_ppl["perplexity"] - cpt_ppl["perplexity"]) / max(base_ppl["perplexity"], 1e-8)

    base_general = generate_baseline(base_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    cpt_general = generate_baseline(cpt_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    comparison = [{"prompt": p, "base_output": b["generated_text"], "cpt_output": c["generated_text"], "verdict": "Retained"} for p, b, c in zip(GENERAL_PROMPTS, base_general, cpt_general)]

    print(json.dumps({"base": base_ppl, "cpt": cpt_ppl, "ppl_reduction_percent": reduction}, indent=2))
    (OUTPUTS / "forgetting_comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    comparison
else:
    print("Evaluation ready. Set RUN_EVALUATION=True.")

Evaluation ready. Set RUN_EVALUATION=True.


## Part B — Instruction Fine-Tuning and Adapter Comparison

This section trains and compares instruction-tuned adapters.

- First cell loads/creates the instruction dataset and formats training text.
- Second cell trains adapters A, B, and C using the same dataset split.
- Final cell evaluates all trained adapters on the same prompts and writes a comparison table to `outputs/adapter_comparison.json`.

In [ ]:
from transformers import AutoTokenizer

if not DATASET_PATH.exists():
    dataset_stats = build_instruction_dataset(CLEAN_CORPUS, DATASET_PATH, min_pairs=100, train_ratio=0.8)
    print("Dataset created:", dataset_stats)

tokenizer = AutoTokenizer.from_pretrained(str(CPT_DIR if CPT_DIR.exists() else MODEL_ID))
rows = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]

def format_example(row):
    messages = [{"role": "user", "content": row["instruction"]}, {"role": "assistant", "content": row["response"]}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return "### Instruction:\n" + row["instruction"] + "\n\n### Response:\n" + row["response"]

for row in rows:
    row["text"] = format_example(row)

ADAPTER_CONFIGS = {
    "adapter_A": {"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "v_proj"]},
    "adapter_B": {"r": 16, "lora_alpha": 32, "target_modules": ["q_proj", "v_proj"]},
    "adapter_C": {"r": 32, "lora_alpha": 32, "target_modules": ["q_proj", "v_proj", "o_proj"]},
}
print("Prepared", len(rows), "instruction rows")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'[SSL: CERTIFICATE_VERIFY_FA

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/microsoft/biogpt-large/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'[SSL: CERTIFICATE_VERIFY_FA

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
def train_one_adapter(name, cfg, train_rows, eval_rows):
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from trl import SFTTrainer

    model_source = str(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
    runtime = detect_runtime()
    quantized = False

    if runtime["device"] == "cuda":
        try:
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            model = AutoModelForCausalLM.from_pretrained(model_source, quantization_config=bnb, device_map="auto")
            model = prepare_model_for_kbit_training(model)
            quantized = True
        except Exception as exc:
            print("4-bit path unavailable, falling back to full precision LoRA:", exc)
            model = AutoModelForCausalLM.from_pretrained(model_source, torch_dtype=torch.float16, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(model_source)
        model = model.to(runtime["device"])

    available = {module_name.split(".")[-1] for module_name, _ in model.named_modules()}
    resolved_targets = ["out_proj" if target == "o_proj" and "o_proj" not in available and "out_proj" in available else target for target in cfg["target_modules"]]
    missing = [target for target in resolved_targets if target not in available]
    if missing:
        raise ValueError(f"Adapter {name} target modules not found: {missing}")

    lora = LoraConfig(
        r=cfg["r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=resolved_targets,
    )
    model = get_peft_model(model, lora)

    args = TrainingArguments(
        output_dir=str(ADAPTER_ROOT / name),
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        fp16=(runtime["device"] == "cuda" and runtime["dtype"] == "float16"),
        bf16=(runtime["device"] == "cuda" and runtime["dtype"] == "bfloat16"),
        report_to="none",
        remove_unused_columns=False,
    )

    train_ds = Dataset.from_list(train_rows)
    eval_ds = Dataset.from_list(eval_rows)

    try:
        trainer = SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            processing_class=tokenizer,
            dataset_text_field="text",
            max_seq_length=512,
        )
    except TypeError:
        trainer = SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            tokenizer=tokenizer,
            dataset_text_field="text",
            max_seq_length=512,
        )

    trainer.train()
    trainer.save_model(str(ADAPTER_ROOT / name))
    tokenizer.save_pretrained(str(ADAPTER_ROOT / name))
    return {"adapter": name, "quantized_4bit": quantized, "output": str(ADAPTER_ROOT / name)}


if RUN_QLORA:
    train_rows = [r for r in rows if r["split"] == "train"]
    eval_rows = [r for r in rows if r["split"] == "eval"]
    training_summaries = []
    for name, cfg in ADAPTER_CONFIGS.items():
        training_summaries.append(train_one_adapter(name, cfg, train_rows, eval_rows))
    training_summaries
else:
    print("QLoRA ready. Set RUN_QLORA=True.")

In [ ]:
def evaluate_adapter(adapter_path: Path, prompts: list[str]):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    source = str(CPT_DIR if CPT_DIR.exists() else MODEL_ID)
    runtime = detect_runtime()

    if runtime["device"] == "cuda":
        base = AutoModelForCausalLM.from_pretrained(source, torch_dtype=torch.float16, device_map="auto")
    else:
        base = AutoModelForCausalLM.from_pretrained(source).to(runtime["device"])

    model = PeftModel.from_pretrained(base, str(adapter_path))
    outputs = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt")
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        outputs.append(tokenizer.decode(ids[0], skip_special_tokens=True))
    return outputs


EVAL_PROMPTS = [
    "Explain why repeated blood-pressure measurements are useful in hypertension.",
    "What is the difference between sensitivity and specificity?",
    "Why does antimicrobial stewardship matter?",
]

if RUN_ADAPTER_EVAL:
    comparison = {}
    for name in ADAPTER_CONFIGS:
        path = ADAPTER_ROOT / name
        comparison[name] = evaluate_adapter(path, EVAL_PROMPTS) if path.exists() else ["Adapter not found"] * len(EVAL_PROMPTS)

    table = [{"prompt": prompt, **{name: comparison[name][idx] for name in ADAPTER_CONFIGS}} for idx, prompt in enumerate(EVAL_PROMPTS)]
    (OUTPUTS / "adapter_comparison.json").write_text(json.dumps(table, indent=2), encoding="utf-8")
    table
else:
    print("Adapter evaluation ready. Set RUN_ADAPTER_EVAL=True after training.")